## Importing modules

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display

### Importing from files

In [2]:
import sys
import os

# Add the parent directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# Now import the module
from relativistic_degrees_of_freedom import RelativisticDegreesOfFreedom

In [3]:
# --- Class deals with: evolution of the entropy relativistc degrees of freedom
RelativisticDOF = RelativisticDegreesOfFreedom()

## Relativistc entropy degrees of freedom

Based on work: 1803.01038 (authors of publication 2205.01637 follows exaclt in this way)

In [4]:
def f_p(x):
    return np.exp(-1.04855 * x) * (1 + 1.03757 * x + 0.508630 * x**2 + 0.0893988 * x**3)

def b_p(x):
    return np.exp(-1.03149 * x) * (1 + 1.03317 * x + 0.398264 * x**2 + 0.0648056 * x**3)

def f_s(x):
    return np.exp(-1.04190 * x) * (1 + 1.03400 * x + 0.456426 * x**2 + 0.0595248 * x**3)

def b_s(x):
    return np.exp(-1.03365 * x) * (1 + 1.03397 * x + 0.342548 * x**2 + 0.0506182 * x**3)

def S_fit(x):
    return 1 + 7/4 * np.exp(-1.0419 * x) * (1 + 1.034 * x + 0.456426 * x**2 + 0.0595249 * x**3)

def g_starS(T):
    """
    Computes g_{*S}(T) based on the temperature regions defined in the paper.
    
    Parameters:
    T : float or array-like
        Temperature in GeV.
    
    Returns:
    g_starS : float or array-like
        The value of g_{*S} at the given temperature.
    """
    a_i = np.array([
        1, 1.11724E+00, 3.12672E-01, -4.68049E-02,
        -2.65004E-02, -1.19760E-03, 1.82812E-04, 1.36436E-04,
        8.55051E-05, 1.22840E-05, 3.82259E-07, -6.87035E-09
    ])

    b_i = np.array([
        1.43382E-02, 1.37559E-02, 2.92108E-03, -5.38533E-04,
        -1.62496E-04, -2.87906E-05, -3.84278E-06, 2.78776E-06,
        7.40342E-07, 1.17210E-07, 3.72499E-09, -6.74107E-11 
    ])

    c_i = np.array([
        1, 6.07869E-01, -1.54485E-01, -2.24034E-01,
        -2.82147E-02, 2.90620E-02, 6.86778E-03, -1.00005E-03,
        -1.69104E-04, 1.06301E-05, 1.69528E-06, -9.33311E-08
    ])

    d_i = np.array([
        7.07388E+01, 9.18011E+01, 3.31892E+01, -1.39779E+00,
        -1.52558E+00, -1.97857E-02, -1.60146E-01,  8.22615E-05,
        2.02651E-02, -1.82134E-05, 7.83943E-05, 7.13518E-05
    ])
        
    T = np.atleast_1d(T)  # Ensure T is an array
    g_starS_values = np.zeros_like(T, dtype=float)
    
    mask_low = T < 0.120
    mask_high = (T >= 0.120) & (T <= 1e16)
    
    if np.any(mask_low):
        m_e, m_mu, m_pi0, m_piPlus = 0.511e-3, 0.1056, 0.135, 0.140  # Masses in GeV
        m_1, m_2, m_3, m_4 = 0.5, 0.77, 1.2, 2  # Masses in GeV
        g_starS_values[mask_low] = (2.008 + 
                                    1.923 * S_fit(m_e / T[mask_low]) + 
                                    3.442 * f_s(m_e / T[mask_low]) + 
                                    3.468 * f_s(m_mu / T[mask_low]) +
                                    1.034 * b_s(m_pi0/ T[mask_low]) +
                                    2.068 * b_s(m_piPlus/ T[mask_low]) +
                                    4.16 * b_s(m_1/ T[mask_low]) +
                                    30.55 * b_s(m_2/ T[mask_low]) +
                                    90 * b_s(m_3/ T[mask_low]) + 
                                    6209 * b_s(m_4/ T[mask_low])
                                   )
    
    if np.any(mask_high):
        t = np.log(T[mask_high])  # Convert to GeV scale
        sum_ai = np.sum(a_i[:, None] * t**np.arange(len(a_i))[:, None], axis=0)
        sum_bi = np.sum(b_i[:, None] * t**np.arange(len(b_i))[:, None], axis=0)
        sum_ci = np.sum(c_i[:, None] * t**np.arange(len(c_i))[:, None], axis=0)
        sum_di = np.sum(d_i[:, None] * t**np.arange(len(d_i))[:, None], axis=0)
        
        g_starP = sum_ai/sum_bi
        g_starS = g_starP/(1 + sum_ci/sum_di)
        g_starS_values[mask_high] = g_starS
    
    return g_starS_values if len(g_starS_values) > 1 else g_starS_values[0]

In [5]:
# Example usage:
T_values = np.array([1.77/30])  # Temperatures in GeV
print(g_starS(T_values))

14.926972901348597


## Thermall axion

We investigate the Figure 1 from 2205.01637, where we can notice few points: ($m_\mathrm{a}$, $T_{\mathrm{dec}}$). To calculate the extra relativistc degrees of freedom we used (2.14) and fractional axion abundance we used (2.15).

In [6]:
# -------------------- SET FORMULAS FROM PUBLICATION -------------------- #
def calculate_dNeff(T_dec):
    """
    Approximation: Instantaneous axion decoupling.
    
    Parameters:
    T_dec : float or array-like
        Decouple temperature in [GeV].
        
    Returns:
    float or np.ndarray
        Extra relativistic degrees of freedom (ΔNeff).
    """
    # axion decouple
    x_dec = 1  # decouple from the plasma

    # Check if T_dec is array-like (list, tuple, or NumPy array)
    if isinstance(T_dec, (list, tuple, np.ndarray)):
        T_dec = np.array(T_dec)  # Convert to NumPy array for vectorized operations
        # axion_decouple_dof = np.array([RelativisticDOF.compute_decoupling_dof(t, x_dec) for t in T_dec])  # My old approache
        axion_decouple_dof = np.array([g_starS(t/x_dec) for t in T_dec])  # More accurate one
    else:
        axion_decouple_dof = g_starS(T_dec/x_dec)

    # Formula (2.14) from your reference
    dNeff = 0.027 * (axion_decouple_dof / 106.75) ** (-4 / 3)

    return dNeff
    
def calculate_wa(T_dec, m_a):
    """
    Approximation: instantenous axion decouple
    T_dec: decouple temperature in [eV]
    m_a: axion mass in [eV]
    """
    # find extra relativistic degrees of freedom
    dNeff = calculate_dNeff(T_dec)

    # formula (2.15)
    wa = 0.011 * m_a * dNeff**(3/4)
    return wa

In [7]:
# -------------------- TAKE AXION MASS AND DECOUPLE TEMPERATURE -------------------- #
# Define the file path
file_path = "Figure1_points"  # Change to the actual file name

# Initialize storage structures
data_dict = {}
data_dict["m_a"] = np.array([])    # [eV]
data_dict["T_dec"] = np.array([])  # [GeV]
data_dict["dNeff"] = np.array([])
data_dict["w_a"] = np.array([])

# Read the file
with open(file_path, "r") as file:
    for line in file:
        # Skip comments or headers
        if line.startswith("#"):
            continue

        # Split the line into two values (m_a, T_d)
        values = line.strip().split(",")

        # Convert to floats and store
        m_a, T_d = float(values[0]), float(values[1])  # in [eV], [GeV]

        # calculate other values
        dNeff = calculate_dNeff(T_d)
        w_a = calculate_wa(T_d, m_a)
        
        # appending
        data_dict["m_a"] = np.append(data_dict["m_a"], m_a)
        data_dict["T_dec"] = np.append(data_dict["T_dec"], T_d)
        data_dict["dNeff"] = np.append(data_dict["dNeff"], dNeff)
        data_dict["w_a"] = np.append(data_dict["w_a"], w_a)

In [8]:
# Create a DataFrame
df = pd.DataFrame({
    "Axion Mass (m_a) [eV]": data_dict["m_a"],
    "Decouple Temp (T_dec) [GeV]": data_dict["T_dec"],
    "ΔNeff (dNeff)": data_dict["dNeff"],
    "Axion Abundance (w_a)": data_dict["w_a"]
})

display(df)

,Axion Mass (m_a) [eV],Decouple Temp (T_dec) [GeV],ΔNeff (dNeff),Axion Abundance (w_a)
0,0.000100,0.051642,0.388976,5.443958e-07
1,0.001001,0.056162,0.378082,5.307971e-06
2,0.010064,0.050765,0.391299,5.477058e-05
3,0.100721,0.067105,0.356963,5.116582e-04
4,1.003198,0.157676,0.164976,2.856574e-03
5,2.989257,0.279136,0.079109,4.904845e-03
6,9.992022,1.135063,0.047231,1.113561e-02
7,30.060043,0.365204,0.067630,4.385195e-02
8,100.480022,0.801828,0.051183,1.189366e-01


In [9]:
# -------------------- CROSS CHECK ------------------------------------------------------------------- #
# We take values from the axes of Figure 1. We compare those one which ones we calculate
# The only error here witch can be take into account is the differences in calculation
# relativistc degrees of freedom (our caluclation and guys from the publication)
T_dec_arr = np.array([1e-5, 1e-3, 1e-1, 1e1, 1e3])  # [GeV]
dNeff_computed = calculate_dNeff(T_dec_arr)
dNeff_reference = [2.202, 0.590, 0.305, 0.039, 0.028]
relative_diff = abs(dNeff_reference - dNeff_computed)/dNeff_reference * 100

# Create a DataFrame
dCheck = pd.DataFrame({
    "Decouple Temp (T_dec) [GeV]": T_dec_arr,
    "ΔNeff computed": dNeff_computed,
    "ΔNeff reference": dNeff_reference,
    "relative error [%]": relative_diff
})

display(dCheck)

,Decouple Temp (T_dec) [GeV],ΔNeff computed,ΔNeff reference,relative error [%]
0,0.00001,2.203858,2.202,0.084365
1,0.00100,0.591036,0.590,0.175552
2,0.10000,0.304916,0.305,0.027578
3,10.00000,0.039380,0.039,0.974747
4,1000.00000,0.027905,0.028,0.339900


In [10]:
g_starS(10**4)

104.13136691101595

In [11]:
g_starS(1.777/30)

14.9413342440807